# Double DQN Training

Trains a Double DQN agent (SB3 default) to trade EMERALDS & TOMATOES.
MLP architecture: `19 features x 2 products -> 64 -> 64 -> 9 actions`.

In [ ]:
import os, sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
sys.path.insert(0, PROJECT_ROOT)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir:     {DATA_DIR}')

In [ ]:
import numpy as np
import pandas as pd
from RLM.shared.data_loader import load_prices, load_trades

prices_df = load_prices(data_dir=DATA_DIR)
trades_df = load_trades(data_dir=DATA_DIR)

print(f'Prices: {len(prices_df)} rows')
print(f'Trades: {len(trades_df)} rows')
print(f'Products: {prices_df["product"].unique()}')
print(f'Days: {sorted(prices_df["day"].unique())}')

In [ ]:
from RLM.shared.config import PRODUCTS, TRAIN_CONFIG, DQN_CONFIG, NETWORK_CONFIG
from RLM.shared.features import FeatureComputer, compute_features_from_row, fit_normalizer
from RLM.shared.env import TradingEnv

def compute_normalization_params(prices_df, trades_df, products, days):
    all_features = {p: [] for p in products}
    for day in days:
        for product in products:
            fc = FeatureComputer(product)
            day_prices = prices_df[(prices_df['day'] == day) & (prices_df['product'] == product)]
            day_prices = day_prices.sort_values('timestamp').reset_index(drop=True)
            day_trades = trades_df[trades_df['symbol'] == product].sort_values('timestamp')
            for _, row in day_prices.iterrows():
                ts = row['timestamp']
                ts_trades = day_trades[day_trades['timestamp'] == ts]
                trades = list(zip(ts_trades['price'], ts_trades['quantity'])) if len(ts_trades) > 0 else None
                features = compute_features_from_row(row, fc, position=0, trades=trades)
                all_features[product].append(features)
    combined = np.vstack([np.array(v) for v in all_features.values()])
    return fit_normalizer(combined)

feat_means, feat_stds = compute_normalization_params(
    prices_df, trades_df, PRODUCTS, TRAIN_CONFIG['train_days']
)
print(f'Normalization computed: {len(feat_means)} features')

## Train

Adjust `TOTAL_TIMESTEPS`, `SEED`, `LEARNING_RATE` below.

In [ ]:
TOTAL_TIMESTEPS = 100_000
SEED = 42
LEARNING_RATE = 1e-3

from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback

train_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG['train_days'][0],
    augment=True, seed=SEED,
)
eval_env = TradingEnv(
    prices_df=prices_df, trades_df=trades_df,
    products=PRODUCTS, day=TRAIN_CONFIG['eval_days'][0],
    augment=False, seed=SEED + 1,
)
for product in PRODUCTS:
    train_env.feature_computers[product].feature_means = feat_means
    train_env.feature_computers[product].feature_stds = feat_stds
    eval_env.feature_computers[product].feature_means = feat_means
    eval_env.feature_computers[product].feature_stds = feat_stds

MODEL_DIR = os.path.join(PROJECT_ROOT, 'RLM', 'double_dqn', 'policy_weights')
os.makedirs(MODEL_DIR, exist_ok=True)

eval_callback = EvalCallback(
    eval_env, best_model_save_path=MODEL_DIR, log_path=MODEL_DIR,
    eval_freq=max(TOTAL_TIMESTEPS // 20, 1000), n_eval_episodes=5,
    deterministic=True, verbose=1,
)

model = DQN(
    'MlpPolicy', train_env,
    learning_rate=LEARNING_RATE, buffer_size=DQN_CONFIG['buffer_size'],
    learning_starts=DQN_CONFIG['learning_starts'], batch_size=DQN_CONFIG['batch_size'],
    gamma=DQN_CONFIG['gamma'], exploration_fraction=DQN_CONFIG['exploration_fraction'],
    exploration_initial_eps=DQN_CONFIG['exploration_initial_eps'],
    exploration_final_eps=DQN_CONFIG['exploration_final_eps'],
    target_update_interval=DQN_CONFIG['target_update_interval'],
    train_freq=DQN_CONFIG['train_freq'], gradient_steps=DQN_CONFIG['gradient_steps'],
    policy_kwargs=dict(net_arch=NETWORK_CONFIG['hidden_sizes']),
    device='auto', verbose=1, seed=SEED,
)

print(f'Device: {model.device}')
print(f'Training for {TOTAL_TIMESTEPS:,} timesteps...')
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback, progress_bar=True)

model.save(os.path.join(MODEL_DIR, 'final_model'))
np.savez(os.path.join(MODEL_DIR, 'norm_params.npz'), feature_means=feat_means, feature_stds=feat_stds)
print('Training complete!')

## Evaluate

In [ ]:
n_eval_episodes = 10
episode_pnls = []

for ep in range(n_eval_episodes):
    obs, info = eval_env.reset()
    total_reward, done = 0, False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        done = terminated or truncated
    episode_pnls.append(info.get('pnl', total_reward))
    print(f'Episode {ep+1}: PnL={episode_pnls[-1]:.2f}, Positions={info.get("positions", {})}')

pnls = np.array(episode_pnls)
print(f'\nMean PnL: {pnls.mean():.2f}  Std: {pnls.std():.2f}  Min: {pnls.min():.2f}  Max: {pnls.max():.2f}')
if pnls.std() > 0:
    print(f'Sharpe: {pnls.mean() / pnls.std():.2f}')

## Export Weights

Extracts PyTorch weights to `.npz` for the pure-numpy submission.

In [ ]:
weights = {}
layer_idx = 0
for name, param in model.q_net.named_parameters():
    tensor = param.detach().cpu().numpy()
    print(f'  {name}: {tensor.shape}')
    if 'weight' in name:
        weights[f'W{layer_idx}'] = tensor
    elif 'bias' in name:
        weights[f'B{layer_idx}'] = tensor
        layer_idx += 1

weights['feature_means'] = feat_means
weights['feature_stds'] = feat_stds

export_path = os.path.join(MODEL_DIR, 'exported_weights.npz')
np.savez(export_path, **weights)
print(f'\nSaved to: {export_path} ({os.path.getsize(export_path) / 1024:.1f} KB)')

from RLM.shared.numpy_policy import NumpyMLP
numpy_model = NumpyMLP(weights_path=export_path)
action, q_vals = numpy_model.predict(np.random.randn(38).astype(np.float32), normalize=False)
print(f'Verification: action={action}, Q-values={q_vals[:3]}...')